# 02 — MediaPipe feature extraction + QC (Day 2 Step 1)

Runs exactly the same code as `uv run python -m bdsl.cli extract --data pilot50 --mode video` (see `CLAUDE.md` → COMMANDS).

- **Mode:** VIDEO (owner decision, ADR-0023). The pilot feature set is `feat_ver fa89892d4908`.
- **Scope:** pilot50 only. Full-set extraction needs the owner's go, and the CLI refuses other data sets.
- **Inputs:** the 50 pilot *copies* in `dataset/pilot50_v1/videos/`, read only; their SHA-256 is checked before and after.
- **Privacy:** no raw face mesh and no image data are stored (ADR-0017); every `.npz` is checked.
- **Resume:** clips whose `.npz` already carries the current `feat_ver` are skipped, so a re-run only does what is missing.
- **Hand sides:** MediaPipe's "left/right hand" may not be the signer's real left/right (open_questions G8).

In [ ]:
from bdsl.cli import main
from bdsl.config import load_config

cfg = load_config(data="pilot50")
print("env:", cfg["_meta"]["env"], "| videos:", cfg["data"]["video_root"], "| features:", cfg["paths"]["features_dir"])

In [ ]:
# Extraction (VIDEO mode) + QC + privacy check; one worker process per clip.
print("exit code:", main(["extract", "--data", "pilot50", "--mode", "video"]))

In [ ]:
# QC summary of the VIDEO-mode feature set (values only; thresholds are open question G9).
import json
from pathlib import Path

import pandas as pd

from bdsl.features.qc import summarise

video_dirs = [d for d in Path(cfg["paths"]["features_dir"]).iterdir()
              if (d / "feature_meta.json").exists()
              and json.loads((d / "feature_meta.json").read_text(encoding="utf-8"))["mode"] == "video"]
for d in video_dirs:
    s = summarise(pd.read_csv(d / "qc.csv"))
    print(d.name, "| clips:", s["clips"], "| frames:", s["frames"])
    for key in ("cov_left_hand", "cov_right_hand", "cov_both_hands", "cov_any_hand", "hand_side_agree_mean"):
        print(f"  {key:<22} mean {s[key]['mean']}  median {s[key]['median']}")